# Intermolecular Feature Extraction for Fasudil–α-Synuclein MD Analysis

This notebook extracts intermolecular features from the fasudil-bound α-synuclein molecular dynamics trajectory for downstream dimensionality reduction and kinetic modeling. The generated feature matrices describe different aspects of ligand–protein interaction over time, including residue-level ligand proximity, electrostatic/charged-group distances, hydrophobic atom-pair distances, and hydrogen-bond donor–acceptor distances. These saved `.npy` files can then be used as input features for PCA, TICA, clustering, contact-probability analysis, and Markov State Model construction.

In this workflow, the trajectory is first loaded with MDTraj and basic system information is printed to verify topology, residue indexing, and frame count. The notebook then calculates several classes of intermolecular descriptors between fasudil and protein residues/atoms and saves each feature matrix separately for later analysis.

**Outputs generated by this notebook:**

- `distance_matrix_fasudil.npy`: residue-level ligand–protein distance features.
- `charge_matrix_fasudil.npy`: distances between charged ligand atoms and oppositely charged protein groups.
- `hphob_matrix_fasudil.npy`: hydrophobic carbon–carbon ligand–protein distance features.
- `hbond_matrix_fsudil.npy`: hydrogen-bond donor/acceptor distance features.



## Import required libraries

This cell imports the Python packages used for trajectory handling, numerical analysis, plotting, and data storage. `mdtraj` is the main library used to load the MD trajectory and compute distances/contacts, while NumPy and pandas support array manipulation and data organization.


In [1]:
import mdtraj as md
import os
import sys
import numpy as np
import scipy as sp
from scipy import optimize
from scipy.optimize import leastsq
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import math
import itertools    
from numpy import log2, zeros, mean, var, sum, loadtxt, arange, \
                  array, cumsum, dot, transpose, diagonal, floor
from numpy.linalg import inv, lstsq
import pyblock
import pandas as pd

np.set_printoptions(formatter={'float': lambda x: "{0:0.3f}".format(x)})


## Check the current working directory

This cell prints the current directory from which the notebook is being run. This is useful for confirming where output files will be written and for troubleshooting relative file paths.


In [1]:
!pwd

/data/asn/paper/example_ipython_notebooks/Feature_extraction


## Load the fasudil trajectory and topology

This cell defines the input PDB topology and XTC trajectory files, loads the trajectory with MDTraj, centers the coordinates, and stores basic trajectory metadata such as topology, first frame, last frame, and total frame count. The commented section appears to be an optional topology-reindexing workaround that can be used if residue ordering or indexing needs to be corrected before loading the trajectory.


In [8]:
pdb = '/data/asn/biorxiv2021-6626290-no-water-glue/fasudil.pdb'
rep0 = '/data/asn/biorxiv2021-6626290-no-water-glue/fasudil.xtc'

# table,bonds = md.load(pdb).topology.to_dataframe()
# first = table[table["resSeq"]==121].to_numpy()
# first[:,0] = np.arange(1, 1+len(first))
# second = table[table["resSeq"]!=121].to_numpy()
# second[:,0] = np.arange(first[-1,0]+1, first[-1,0]+1 + len(second))
# final = np.concatenate([first, second], axis=0)
# df = pd.DataFrame(data = final, columns = table.columns)
# top_fix = md.Topology.from_dataframe(df, bonds)
#trj = md.load(rep0, top =top_fix)
trj = md.load(rep0, top =pdb)
trj.center_coordinates()
top = trj.topology
first_frame = 0
last_frame = trj.n_frames
n_frames = trj.n_frames

## Inspect system composition and residue indexing

This cell extracts residue numbers, residue names, residue indices, and protein-only residue information from the topology. It then prints a system summary, including the number of atoms, residues, protein residues, frames, and the residue sequence/indexing. This verification step is important because all later feature calculations depend on correct residue and atom indexing.


In [9]:
nres = []
for res in trj.topology.residues:
    nres.append(res.resSeq)
sequence = (' %s' % [residue for residue in trj.topology.residues])
resname = (' %s' % [residue.name for residue in trj.topology.residues])
resindex = (' %s' % [residue.index for residue in trj.topology.residues])
prot_top = top.subset(top.select('protein'))
prot_res = []
for res in prot_top.residues:
    prot_res.append(res.resSeq)
prot_resname = (' %s' % [residue.name for residue in prot_top.residues])
residues = len(set(prot_res))

#log = open("/Users/paulrobustelli/Desktop/Sa_calc.log", "w")
print("** SYSTEM INFO **\n")
print("Number of atoms: %d\n" % trj.n_atoms)
print("Number of residues: %d\n" % len(set(nres)))
print("Number of protein residues: %d\n" % len(set(prot_res)))
print("Number of frames: %d\n" % trj.n_frames)
print("Starting frame: %d\n" % first_frame)
print("Last frame: %d\n" % last_frame)
print("sequence: %s\n" % sequence)
print("residue names: %s\n" % resname)
print("residue index: %s\n" % resindex)


residue_offset = 0
prot_res_renum = np.asarray(prot_res)+residue_offset
residue_number = range(0, residues)
residue_number_offsetres = range(residue_offset, residue_offset+residues)



** SYSTEM INFO **

Number of atoms: 338

Number of residues: 21

Number of protein residues: 20

Number of frames: 1100889

Starting frame: 0

Last frame: 1100889

sequence:  [ASP121, ASN122, GLU123, ALA124, TYR125, GLU126, MET127, PRO128, SER129, GLU130, GLU131, GLY132, TYR133, GLN134, ASP135, TYR136, GLU137, PRO138, GLU139, ALA140, ASP121, <1>1]

residue names:  ['ASP', 'ASN', 'GLU', 'ALA', 'TYR', 'GLU', 'MET', 'PRO', 'SER', 'GLU', 'GLU', 'GLY', 'TYR', 'GLN', 'ASP', 'TYR', 'GLU', 'PRO', 'GLU', 'ALA', 'ASP', '<1>']

residue index:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]



## Compute residue-level ligand–protein distance features

This cell defines a helper function, `distance_matrix`, that computes pairwise distances either between residues or between atoms across all frames of the trajectory. Here it is used to calculate distances between protein residues `0–19` and the ligand residue `20`. The resulting distance matrix stores one feature per residue–ligand pair over time. A binary contact version is also created using a 0.5 nm cutoff, although the continuous distance matrix is the object saved in the next cell.


In [7]:
def distance_matrix(sel1,sel2,offset1,offset2,traj,measure,periodic):
    ''' RETURNS: dmat,np.array(pairs),np.array(pairs_index),index,columns,len(sel1)..(x),len(sel2)....(y)
    '''
    pair_distances = []
    pairs = []
    pairs_index = []
    if measure == "residues":
        index = [traj.topology.residue(i) for i in sel1]
        columns = [traj.topology.residue(j) for j in sel2]
        for i in sel1:
            for j in sel2:
                pairs.append("{},{}".format(traj.topology.residue(i),traj.topology.residue(j)))
                pairs_index.append([i+offset1,j+offset2])
                if i==j:
                    dist = np.zeros(traj.n_frames)
                    pair_distances.append(dist)
                else:
                    dist = md.compute_contacts(traj,[[i,j]],periodic=periodic)[0][:,0]
                    pair_distances.append(dist)
    if measure == "atoms":
        index = [traj.topology.atom(i) for i in sel1]
        columns = [traj.topology.atom(j) for j in sel2]
        for i in sel1:
            for j in sel2:
                pairs.append("{},{}".format(traj.topology.atom(i),traj.topology.atom(j)))
                pairs_index.append([i+offset1,j+offset2])
                if i==j:
                    dist = np.zeros(traj.n_frames)
                    pair_distances.append(dist)
                else:
                    dist = md.compute_distances(traj,[[i,j]],periodic=periodic)[:,0]
                    pair_distances.append(dist)
    dist_feat_arr = np.stack(pair_distances,axis=1)
    return dist_feat_arr,np.array(pairs),np.array(pairs_index),index,columns,np.array([len(sel1),len(sel2)])
dmat, pairs, pairs_idx, index, col, xy = distance_matrix(np.arange(0,20),[20], 0,0,trj,"residues", True)
dmat_bin = np.where(dmat<.5,1,0)

## Save residue-level distance matrix

This cell saves the continuous residue-level ligand–protein distance matrix as `distance_matrix_fasudil.npy`. This file can be used later as an intermolecular feature set for PCA, TICA, clustering, or MSM construction.


In [4]:
np.save("distance_matrix_fasudil.npy", dmat)

## Compute charged-group distance features

This cell defines charged ligand atoms and selects positively and negatively charged protein groups from the topology. For fasudil, the ligand positive charge is assigned to atom index `296`, while negatively charged protein groups are selected from acidic side chains and terminal oxygen atoms. The cell then builds ligand-positive/protein-negative atom pairs and computes their distances across the trajectory. These features capture possible electrostatic interactions between fasudil and charged α-synuclein residues.


In [ ]:
#Select Ligand Charge Groups
#Ligand Charged atom is N-296
Ligand_Pos_Charges=[296]
Ligand_Neg_Charges=[]

def add_charge_pair(pairs,pos,neg,contact_prob):
    if pos not in pairs: 
        pairs[pos] = {} 
    if neg not in pairs[pos]:
        pairs[pos][neg] = {}
    pairs[pos][neg] = contact_prob

#Select Protein Charge Groups
#Add Apropriate HIS name if there is a charged HIE OR HIP in the structure 
Protein_Pos_Charges=top.select("(resname ARG and name CZ) or (resname LYS and name NZ) or (resname HIE and name NE2) or (resname HID and name ND1)")
#Protein_Neg_Charges=[]
Protein_Neg_Charges=top.select("(resname ASP and name CG) or (resname GLU and name CD) or (name OXT) or (resname NASP and name CG)")
neg_res=[]
pos_res=[]
                               
for i in Protein_Neg_Charges:
    neg_res.append(top.atom(i).residue.resSeq)

for i in Protein_Pos_Charges:
    pos_res.append(top.atom(i).residue.resSeq)                               
                               
print("Negatively Charged Residues:", neg_res)
print("Posiitively Charged Residues", pos_res)

charge_pairs_ligpos=[]                      
for i in Ligand_Pos_Charges:
    for j in Protein_Neg_Charges:              
        charge_pairs_ligpos.append([i,j])
        pos=top.atom(i)
        neg=top.atom(j) 

contact_1  = md.compute_distances(trj, charge_pairs_ligpos)
#contact_2  = md.compute_distances(trj, charge_pairs_ligneg)

## Save charged-group distance matrix

This cell saves the ligand–protein charged-group distance matrix as `charge_matrix_fasudil.npy`. This matrix is intended to represent electrostatic interaction features for downstream analysis.


In [ ]:
np.save("charge_matrix_fasudil.npy", contact_1)

## Compute hydrophobic ligand–protein distance features

This cell selects ligand carbon atoms and protein carbon atoms as a simple hydrophobic-contact feature definition. It prints the selected atoms for inspection, builds all pairwise protein-carbon/ligand-carbon combinations, and computes their distances across the trajectory. The resulting matrix describes hydrophobic proximity between fasudil and protein atoms over time.


In [2]:
# Calculate Hydrophobic contacts
ligand_hphob = top.select("resid 20 and type C")
protein_hphob = top.select("protein and element C")


ligand_hphob_atoms = []
for atom in ligand_hphob:
    ligand_hphob_atoms.append(top.atom(atom))

protein_hphob_atoms = []
for atom in protein_hphob:
    protein_hphob_atoms.append(top.atom(atom))

print(ligand_hphob)
print(ligand_hphob_atoms)

print(protein_hphob)
print(protein_hphob_atoms)


def add_contact_pair(pairs, a1, a2, a1_id, a2_id, prot_res, contact_prob):
    if prot_res not in pairs:
        pairs[prot_res] = {}
    if a2 not in pairs[prot_res]:
        pairs[prot_res][a2] = {}
    if a1_id not in pairs[prot_res][a2]:
        pairs[prot_res][a2][a1_id] = contact_prob


hphob_pairs = []
for j in ligand_hphob:
    for i in protein_hphob:
        hphob_pairs.append([i, j])


contact = md.compute_distances(trj, hphob_pairs)
contacts = np.asarray(contact).astype(float)

## Save hydrophobic distance matrix

This cell saves the hydrophobic carbon–carbon distance matrix as `hphob_matrix_fasudil.npy`. This output can be used to quantify hydrophobic ligand–protein contacts during later dimensionality reduction or kinetic analysis.


In [3]:
np.save("hphob_matrix_fasudil.npy", contacts)

## Compute hydrogen-bond distance features

This cell defines ligand hydrogen-bond donor atom pairs and uses `print_donors_acceptors` to identify donor–acceptor combinations in the first trajectory frame using geometric criteria. It then computes donor–acceptor distances across all frames. These distances can be used to monitor possible ligand–protein hydrogen-bond interactions over the simulation.


In [4]:
# Definie Hydrogen Bond Donors in EPI-002
lig_hbond_donors = [[296, 331], [296, 318]]
hbonds = print_donors_acceptors(
    trj[0], angle_cutoff=150, distance_cutoff=0.35, lig_donor_index=lig_hbond_donors)
dist = md.compute_distances(trj, hbonds[:, [0,2]], periodic=False)

## Save hydrogen-bond distance matrix

This cell saves the hydrogen-bond donor–acceptor distance matrix as `hbond_matrix_fsudil.npy`. This file stores the hydrogen-bond-related feature set for later analysis. Note that the filename appears to use `fsudil` rather than `fasudil`, which may be worth standardizing if the workflow is being cleaned for GitHub.


In [5]:
np.save("hbond_matrix_fsudil.npy", dist)